AIRBNB PROJECT - ARTIFICIAL INTELLIGENCE

In [18]:
import pandas as pd

# File paths (adjust if needed)
listings = pd.read_csv("/Users/talhamurtaza/Downloads/AI/listings.csv")
calendar = pd.read_csv("/Users/talhamurtaza/Downloads/AI/calendar.csv")
reviews = pd.read_csv("/Users/talhamurtaza/Downloads/AI/reviews.csv")

# Preview
print(listings.head())
print(calendar.head())
print(reviews.head())

        id                           listing_url       scrape_id last_scraped  \
0   241032   https://www.airbnb.com/rooms/241032  20160104002432   2016-01-04   
1   953595   https://www.airbnb.com/rooms/953595  20160104002432   2016-01-04   
2  3308979  https://www.airbnb.com/rooms/3308979  20160104002432   2016-01-04   
3  7421966  https://www.airbnb.com/rooms/7421966  20160104002432   2016-01-04   
4   278830   https://www.airbnb.com/rooms/278830  20160104002432   2016-01-04   

                                  name  \
0         Stylish Queen Anne Apartment   
1   Bright & Airy Queen Anne Apartment   
2  New Modern House-Amazing water view   
3                   Queen Anne Chateau   
4       Charming craftsman 3 bdm house   

                                             summary  \
0                                                NaN   
1  Chemically sensitive? We've removed the irrita...   
2  New modern house built in 2013.  Spectacular s...   
3  A charming apartment that sits at

In [19]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from mlflow.tracking import MlflowClient

# ------------------------------------------------
# 1. Set MLflow Tracking
# ------------------------------------------------
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Airbnb Price Prediction")

# ------------------------------------------------
# 2. Load Data
# ------------------------------------------------
listings = pd.read_csv("/Users/talhamurtaza/Downloads/AI/listings.csv")
calendar = pd.read_csv("/Users/talhamurtaza/Downloads/AI/calendar.csv")
reviews = pd.read_csv("/Users/talhamurtaza/Downloads/AI/reviews.csv")

# ------------------------------------------------
# 3. Basic Preprocessing
# ------------------------------------------------

# Rename id column if needed
if "id" in listings.columns:
    listings = listings.rename(columns={"id": "listing_id"})

# Clean price column
listings["price"] = listings["price"].replace('[\$,]', '', regex=True).astype(float)

# Aggregate calendar (availability rate)
calendar["available"] = calendar["available"].map({'t': 1, 'f': 0})
calendar_agg = calendar.groupby("listing_id")["available"].mean().reset_index()
calendar_agg.rename(columns={"available": "availability_rate"}, inplace=True)

# Aggregate reviews (count)
reviews_agg = reviews.groupby("listing_id").size().reset_index(name="review_count")

# Merge all
df = listings.merge(calendar_agg, on="listing_id", how="left")
df = df.merge(reviews_agg, on="listing_id", how="left")

# Fill missing values
df["availability_rate"].fillna(0, inplace=True)
df["review_count"].fillna(0, inplace=True)

# Select simple numeric features
features = ["bedrooms", "bathrooms", "availability_rate", "review_count"]
df = df.dropna(subset=features + ["price"])

X = df[features]
y = df["price"]

# ------------------------------------------------
# 4. Train Model + MLflow Logging
# ------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

with mlflow.start_run() as run:

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)

    # Log parameters
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 100)

    # Log metric
    mlflow.log_metric("rmse", rmse)

    # Log model
    mlflow.sklearn.log_model(model, "model")

    run_id = run.info.run_id

print("Run ID:", run_id)

# ------------------------------------------------
# 5. Register Model
# ------------------------------------------------
model_uri = f"runs:/{run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="AirbnbPriceModel"
)

print("Registered Model Version:", registered_model.version)

/var/folders/yh/k856xb193pq0qgny22xpdmn80000gn/T/ipykernel_90385/3693404177.py:47: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["availability_rate"].fillna(0, inplace=True)
/var/folders/yh/k856xb193pq0qgny22xpdmn80000gn/T/ipykernel_90385/3693404177.py:48: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alw

🏃 View run funny-grub-495 at: http://127.0.0.1:5000/#/experiments/626854755944770602/runs/2f63533f30a64e01bcc7c47234988ad1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/626854755944770602
Run ID: 2f63533f30a64e01bcc7c47234988ad1


Registered model 'AirbnbPriceModel' already exists. Creating a new version of this model...
2026/03/03 08:52:00 WARNING mlflow.tracking._model_registry.fluent: Run with id 2f63533f30a64e01bcc7c47234988ad1 has no artifacts at artifact path 'model', registering model based on models:/m-533e16bf7a124b40b158eec39fde5e6b instead
2026/03/03 08:52:00 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: AirbnbPriceModel, version 2
Created version '2' of model 'AirbnbPriceModel'.


Registered Model Version: 2
